# Lab 1 : From an HR policy to a search

*Week 3 · Utrains LLMOps 8 Week Course*

Run each cell from the top. Read the printed output before you run the next cell.


## What we are achieving in this lab

In Weeks 1 and 2 you sent a question to a **language model** (Ollama or Claude). A language model is  trained on public text. It does not contain your company's **HR policy**.

An HR policy is the internal document employees use for rules such as reimbursements, time off, and parental leave. At a company, people ask those questions every day. If you send them to the model with no policy text, the model cannot look up the real rule. It will guess. That is how a wrong expense or leave answer reaches an employee.

This week's method is: find the relevant paragraphs in *your* HR policy file, then (in Lab 2) put those paragraphs in the prompt so the model answers from them. That method is called **RAG**  Retrieval Augmented Generation. The three letters are three steps, in this order:

 **Retrieval** = search your files for the pieces that match the question
 **Augmentation** = add those retrieved pieces to the prompt, so the model can read them before it writes
 **Generation** = the language model writes the answer


The path in code, in this order:
| Step | Plain meaning | This lab? |
|------|----------------|-----------|
| **Load** | Open `hr_policy.txt` and read it into Python as one string. | Yes |
| **Split** | Cut that string into smaller pieces called **chunks**, so one piece is about one topic. | Yes |
| **Embed** | Turn a piece of text into a **vector** (a list of numbers for meaning). | Yes |
| **Store** | Save those chunk vectors in a vector store (the index). | Yes |
| **Retrieve** | For a new question, return the closest stored chunks. | Yes |
| **Augment** | Add those chunks to the prompt. | Lab 2 |
| **Generate** | The language model writes the answer from that prompt. | Lab 2 |

**What you will do**

1. Cut the HR policy into chunks and see why size matters.
2. Turn text into a **vector** (an embedding) with OpenAI.
3. **Store** those chunk vectors in a simple vector store (no extra install).
4. Ask a question, embed it, and **retrieve** the best matching chunks.

You do not write the similarity math yourself. The vector store does that search for you  the same idea as production.

**Before you start.** Finish Weeks 1 and 2. Copy `.env.example` to `.env` in this folder. Paste `OPENAI_API_KEY` before Step 4. Full setup is in [README.md](./README.md).

**Cost.** A few embedding API calls. Fractions of a cent.


### Step 1. Load the HR policy

**Load** means: open the file and read all of its text into a Python string. The file `hr_policy.txt` is in the same folder as this notebook. It is a short company HR policy: accounts, reimbursements, time off, and parental leave. Print it once so you know what you will cut in the next steps.


In [1]:
with open("hr_policy.txt", encoding="utf-8") as f:
    POLICY = f.read()

print("characters:", len(POLICY))
print()
print(POLICY)


characters: 1502

# HR Policy

This is the official HR policy. Use it for account setup, expense reimbursements, time off, and parental leave.

## Section 1: Setting Up Your Account
To set up your account, visit the company portal at portal.example.com.
Click the "Sign Up" button. You will receive a confirmation email within
five minutes. If you do not see the email, check your spam folder.
The portal supports two factor authentication, which we strongly recommend.

## Section 2: Your First Week
In your first week, your manager will walk you through three things:
the team rituals, the on-call calendar, and the deployment pipeline.
You will also meet your buddy, who is your primary point of contact for
non-urgent questions for your first 30 days.

## Section 3: Reimbursements
To submit a reimbursement, log into the finance portal at finance.example.com.
Upload your receipt as a PDF. Reimbursements take 7 to 10 business days.
For travel under $500, no pre-approval is needed. Above $500 r

### Step 2. What does split mean?

The HR policy is one long string. It covers several topics: accounts, reimbursements, time off, parental leave.

**Split** means: cut that one string into a list of smaller strings. In Python, `POLICY.split("...")` looks for a marker and cuts there.

The next cell cuts on `##` (section headings) so you can see that one file becomes several pieces. This is only a demo of the idea. Step 3 is the split we use for RAG: cut by length.


In [3]:
# split turns one string into a list of smaller strings.
# It cuts wherever "\n## " appears in the text.
parts = POLICY.split("\n## ")

print("one file became", len(parts), "pieces")
print()
for part in parts:
    print(part[:80])
    print("---")


one file became 6 pieces

# HR Policy

This is the official HR policy. Use it for account setup, expense r
---
Section 1: Setting Up Your Account
To set up your account, visit the company por
---
Section 2: Your First Week
In your first week, your manager will walk you throug
---
Section 3: Reimbursements
To submit a reimbursement, log into the finance portal
---
Section 4: Time Off
Submit time off requests through the HR portal. We have an u
---
Section 5: Parental Leave
Full-time employees receive 16 weeks of paid parental 
---


### Step 3. Cut the file into chunks

Step 2 listed headings. It did not create the chunks we will embed.

A real document may have no headings, or one section that is too long. So we cut by length, using LangChain's `RecursiveCharacterTextSplitter`.

`RecursiveCharacterTextSplitter` cuts a long string into smaller strings (chunks). It tries to cut at a paragraph break first, then a newline, then a space, so it does not split in the middle of a word.

 **`chunk_size`**  maximum number of characters in one chunk
 **`chunk_overlap`**  how many characters at the end of one chunk are copied onto the start of the next chunk

**Example, overlap = 0** (cut is a hard edge):

```
text:    Pay the receipt. Above 500 needs approval.
chunk 1: Pay the receipt. Above
chunk 2:                  500 needs approval.
```

A search that finds chunk 1 never sees "needs approval".

**Same text, overlap > 0** (the end of chunk 1 starts chunk 2):

```
chunk 1: Pay the receipt. Above 500
chunk 2:                  Above 500 needs approval.
```

"Above 500" sits in both chunks, so the rule is not lost on the cut.

We try three sizes on the same file: **200** (too small), **500** (usable), **2000** (larger than this file, so still one piece).


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_policy(chunk_size: int, chunk_overlap: int) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    # Returns a list of strings. Each string is one chunk of POLICY.
    return splitter.split_text(POLICY)


# Same file, three chunk sizes. Each result is a list of strings.
small = split_policy(200, 40)    # about 200 characters per piece
medium = split_policy(500, 50)   # about 500 characters per piece — we keep this
huge = split_policy(2000, 0)     # larger than the file, so still one piece

print("chunk_size 200 ->", len(small), "pieces")
print(small)
print()

# print("chunk_size 500 ->", len(medium), "pieces")
# print(medium)
# print()

# print("chunk_size 2000 ->", len(huge), "pieces")
# print(huge)


chunk_size 200 -> 11 pieces
['# HR Policy\n\nThis is the official HR policy. Use it for account setup, expense reimbursements, time off, and parental leave.', '## Section 1: Setting Up Your Account\nTo set up your account, visit the company portal at portal.example.com.\nClick the "Sign Up" button. You will receive a confirmation email within', 'five minutes. If you do not see the email, check your spam folder.\nThe portal supports two factor authentication, which we strongly recommend.', '## Section 2: Your First Week\nIn your first week, your manager will walk you through three things:\nthe team rituals, the on-call calendar, and the deployment pipeline.', 'You will also meet your buddy, who is your primary point of contact for\nnon-urgent questions for your first 30 days.', '## Section 3: Reimbursements\nTo submit a reimbursement, log into the finance portal at finance.example.com.\nUpload your receipt as a PDF. Reimbursements take 7 to 10 business days.', 'For travel under $500, no

Look at the three lists you just printed.

 **200**  many short pieces. One rule can be cut in half across two pieces. Then a question may find only half of the rule.
 **500**  fewer, longer pieces. A topic usually fits in one piece. We keep `medium` for the rest of this lab.
 **2000**  one piece. That is the whole file again. Topics are mixed.

Rule of thumb: too small loses part of a rule. Too big mixes every topic. **500** is the usable size for this file.


### Step 4. Load the OpenAI key

The next steps call OpenAI over the internet. OpenAI turns text into a list of numbers. For that to work, you need an API key in a `.env` file in this folder.

1. Copy `.env.example` to `.env` if you have not already.
2. Paste your `OPENAI_API_KEY` into `.env`.
3. Do not commit `.env` to git. It contains a secret.

The text you send to OpenAI is processed on OpenAI's servers, not only on your computer.


In [5]:
from dotenv import load_dotenv

load_dotenv()  # reads .env from this folder


True

### Step 5. An embedding is a vector

An **embedding** is a list of numbers that stands for the meaning of a piece of text. That list is also called a **vector**.

We use OpenAI `text-embedding-3-small` through LangChain. It is cheap (~$0.02 / 1M tokens) and current.



In [6]:
from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

text = "How do I reset my password?"
vec = embeddings.embed_query(text)

print("text   :", text)
print("model  :", EMBED_MODEL)
print("type   :", type(vec).__name__)
print("length :", len(vec), "numbers")
print("first 8:", [round(x, 4) for x in vec[:8]])


text   : How do I reset my password?
model  : text-embedding-3-small
type   : list
length : 1536 numbers
first 8: [0.0176, -0.0457, 0.0298, 0.0219, -0.0486, 0.002, 0.0016, 0.0614]


**Anthropic does not offer an embedding model.** 

Use the **same** embedding model for every chunk you store, and for every question you search with.

Different models produce different lists of numbers for the same sentence.

So do not mix models. Embed the policy chunks with model A. Embed the question with model A too. Then you can compare them.

If you later decide to use a different model, delete the old number lists and create new ones for every chunk with that new model.



An **embedding** is a Python list of floating-point numbers. That list is also called a **vector**.

For OpenAI `text-embedding-3-small`, every vector is always **1536** numbers long. A short word and a long paragraph both become a vector of length 1536. Same length means you can compare any two of them.


In [8]:
short = "password"
long = (
    "To reset your password, open the sign-in page, click Forgot Password, "
    "and check the email we send within five minutes."
)

short_vec = embeddings.embed_query(short)
long_vec = embeddings.embed_query(long)

print(repr(short), "->", len(short_vec), "numbers")
print("paragraph   ->", len(long_vec), "numbers")
print("same length :", len(short_vec) == len(long_vec))


'password' -> 1536 numbers
paragraph   -> 1536 numbers
same length : True


RAG uses two embedding calls.

An **index** is the saved set of your chunks and their vectors. You build it once when the HR policy is ready (or when the chunks change). Later, when someone asks a question, you only embed that one question and search the index.

| When | Method | Input |
|------|--------|--------|
| You build the index (chunks change) | `embed_documents` | many strings (your chunks) |
| A user asks a question | `embed_query` | one string (the question) |

Try both calls below on toy text. In Step 6 you will reuse the **same** `embeddings` object: the vector store will call the `embed_documents` idea for your real `medium` chunks, and in Step 7 search will use the `embed_query` idea for the question.


In [9]:
query_vec = embeddings.embed_query(text)
doc_vecs = embeddings.embed_documents(
    [
        "Click Forgot Password on the sign-in page.",
        "Our offices open at 9am Pacific.",
    ]
)

print("embed_query    : 1 vector of", len(query_vec), "numbers")
print("embed_documents:", len(doc_vecs), "vectors, each", len(doc_vecs[0]), "numbers")


embed_query    : 1 vector of 1536 numbers
embed_documents: 2 vectors, each 1536 numbers


### Step 6. Store the chunk vectors

In Step 5 you learned two calls on an `embeddings` object:

| Step 5 method | What it does |
|---------------|--------------|
| `embed_documents([...])` | Turn **many chunks** into vectors (build the index) |
| `embed_query("...")` | Turn **one question** into a vector (search later) |

You could call `embed_documents(medium)` yourself and keep two lists: chunk text, and vectors. That works, but it is easy to lose track of which vector belongs to which chunk.

So LangChain packages each chunk as a **`Document`**: the chunk text (`page_content`) plus a small label (`metadata`, here the chunk number). Then **`InMemoryVectorStore.from_documents`** takes those Documents and an `embeddings` object. Under the hood it runs the same idea as `embed_documents` on every chunk, and saves **chunk text + vector** together. That saved set is the **index**.

The next cell rebuilds `medium` (Step 3 size) and `embeddings` (Step 5 model), builds the store, then **prints what is inside**  each item’s text and a short look at its vector. Search comes in Step 7.

In this lab the store lives in this notebook's memory only. Restart the kernel, and it is empty. In a company you often use a vector database on disk or in the cloud (Chroma, pgvector, Pinecone, Qdrant). The idea is the same: save chunk + vector, then search later.


In [13]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()  # same as Step 4 — needs OPENAI_API_KEY in .env

# Same file and chunk size as Step 3 (medium = 500 / 50).
with open("hr_policy.txt", encoding="utf-8") as f:
    POLICY = f.read()

medium = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
).split_text(POLICY)

# Same embedding model as Step 5.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Document = chunk text + a small label (chunk number).
docs = []
for i, text in enumerate(medium):
    docs.append(Document(page_content=text, metadata={"chunk": i}))

# from_documents ≈ Step 5 embed_documents on every chunk, then save text + vector.
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)




In [14]:
# Look inside the store. Each item has text + vector (+ metadata).
print("stored items:", len(vectorstore.store))
print()

for item in vectorstore.store.values():
    vec = item["vector"]
    text = item["text"]
    print("--- chunk", item["metadata"]["chunk"], "---")
    print("text  :", text[:150], "..." if len(text) > 150 else "")
    print("vector: length", len(vec), "| first 8:", [round(x, 4) for x in vec[:8]])
    print()

stored items: 4

--- chunk 0 ---
text  : # HR Policy

This is the official HR policy. Use it for account setup, expense reimbursements, time off, and parental leave.

## Section 1: Setting Up ...
vector: length 1536 | first 8: [0.0052, 0.0098, 0.0388, 0.0302, 0.0605, -0.0128, 0.0217, 0.02]

--- chunk 1 ---
text  : ## Section 2: Your First Week
In your first week, your manager will walk you through three things:
the team rituals, the on-call calendar, and the dep ...
vector: length 1536 | first 8: [-0.0237, 0.0495, 0.0269, 0.0133, -0.0473, -0.0303, 0.0331, 0.011]

--- chunk 2 ---
text  : ## Section 3: Reimbursements
To submit a reimbursement, log into the finance portal at finance.example.com.
Upload your receipt as a PDF. Reimbursemen ...
vector: length 1536 | first 8: [-0.0413, 0.0087, 0.0205, -0.0173, -0.0365, -0.0043, 0.0011, 0.0173]

--- chunk 3 ---
text  : ## Section 4: Time Off
Submit time off requests through the HR portal. We have an unlimited PTO
policy with a 2 week minimum s

### Step 7. Ask a question and retrieve the best matches

Step 6 only stored the index. Now we search it.

When you call `retriever.invoke(question)`, the store:

1. Runs the same idea as **`embed_query(question)`**  one vector for the question (same `embeddings` model).
2. Compares that vector to the chunk vectors saved in Step 6 (those came from **`embed_documents`** under the hood).
3. Returns the closest **chunks** (the text), not only the numbers  so you can read them and, in Lab 2, put them in a prompt.

You do not call `embed_query` by hand here. `retriever.invoke` does it for you.

Read the printed chunks. For a train-ticket reimbursement question, the reimbursement section should be near the top.


In [15]:
# k=3 means: return the 3 closest chunks.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

QUESTION = "How do I get reimbursed for a $300 train ticket?"

# Under the hood: embed_query(question), then find closest stored vectors.
hits = retriever.invoke(QUESTION)

print("Question:", QUESTION)
print()
print("Best matching chunks:")
print()
for i, hit in enumerate(hits, start=1):
    print("--- match", i, "  chunk", hit.metadata.get("chunk"), "---")
    print(hit.page_content)
    print()


Question: How do I get reimbursed for a $300 train ticket?

Best matching chunks:

--- match 1   chunk 2 ---
## Section 3: Reimbursements
To submit a reimbursement, log into the finance portal at finance.example.com.
Upload your receipt as a PDF. Reimbursements take 7 to 10 business days.
For travel under $500, no pre-approval is needed. Above $500 requires
your manager and finance team approval.

--- match 2   chunk 0 ---
# HR Policy

This is the official HR policy. Use it for account setup, expense reimbursements, time off, and parental leave.

## Section 1: Setting Up Your Account
To set up your account, visit the company portal at portal.example.com.
Click the "Sign Up" button. You will receive a confirmation email within
five minutes. If you do not see the email, check your spam folder.
The portal supports two factor authentication, which we strongly recommend.

--- match 3   chunk 3 ---
## Section 4: Time Off
Submit time off requests through the HR portal. We have an unlimited PT

Those matches are the paragraphs you would put into the prompt in Lab 2.

Under the hood the store ranks chunks by how close their vectors are to the question vector. You do not need the math formula for that. In production the vector database does the same job.

## What you should be able to explain

- I split a document into chunks before I embed. Too small loses a rule. Too big mixes every topic.
- An embedding is a vector: a fixed-length list of numbers. I use one embedding model for chunks and for questions.
- An index is the saved chunks and their vectors.
- A vector store keeps that index and returns the closest chunks for a question. I call `retriever.invoke`.
- Lab 2 takes those chunks and asks a language model to answer only from them.
